### ***SUMMARY***

Framing divides each audio signal into short, fixed-size sections that overlap. Speech changes over time, but it is nearly stable within each short frame. Feature methods such as magnitude spectra, MFCC, LPCC, and PLP can therefore describe each frame consistently. Fixed-size frames also give the classifier samples with the same dimensions.

The saved array keeps each frame with its source-file path and its frame number within that file. These values identify the recording and preserve the frame sequence. Later steps can reconstruct the time order, examine neighboring frames, and trace a result to its original position. The source-file link also supports labels and prevents data leakage because frames from one recording must stay in the same data group.

Saving the framed data makes preprocessing repeatable and avoids framing every file again during feature extraction. Later notebooks can load one consistent data set, apply windowing and feature extraction, and compare methods on the same frames.

In [ ]:
from pathlib import Path

# In a notebook, Path.cwd() is the folder the notebook runs from.
NOTEBOOK_DIR:Path = Path.cwd()
REPOSITORY_ROOT:Path = NOTEBOOK_DIR

DEFAULT_SOURCE:Path = REPOSITORY_ROOT / "Artists"
DEFAULT_OUTPUT:Path = NOTEBOOK_DIR / "preprocessing-01-framing.npy"

### ***GETTING DIRECTORIES***

In [10]:
import argparse

parser: argparse.ArgumentParser = argparse.ArgumentParser(description="Frame audio files.")
parser.add_argument("--source", type=Path, default=DEFAULT_SOURCE)
parser.add_argument("--output", type=Path, default=DEFAULT_OUTPUT)
args:argparse.Namespace = parser.parse_args([])

print(args)

Namespace(source=PosixPath('/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists'), output=PosixPath('/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/framed_audio.npy'))


### ***GETTING FILE DIRECTORIES***

In [11]:
AUDIO_SUFFIXES:set[str] = {".flac", ".m4a", ".mp3", ".ogg", ".wav"}

source_dir:Path = args.source.resolve()
if not source_dir.is_dir():
    raise FileNotFoundError(f"Source directory does not exist: {source_dir}")
FILES_AUDIO:list[Path] = sorted(
    path
    for path in source_dir.rglob("*")
    if path.is_file() and path.suffix.lower() in AUDIO_SUFFIXES
)
if not FILES_AUDIO:
    raise FileNotFoundError(f"No supported audio FILES_AUDIO found under {source_dir}")

for x in FILES_AUDIO:
    print(x)

/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Dax-Dr-Dre-Ft-Eminem-Forgot-About-Dre-Remix-(TrendyBeatz.com).mp3
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Eminem-Godzilla-ft-Juice-WRLD-(HipHopKit.com).mp3
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Eminem-Mockingbird-(HipHopKit.com).mp3
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Eminem-Snoop-Dogg-From-The-D-2-The-LBC-(HipHopKit.com).mp3
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Eminem-ft-Joyner-Lucas-Lucky-You-(HipHopKit.com).mp3
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Eminem-ft-NF-Why-(HipHopKit.com).mp3
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/EminemVenom.mp3
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Eminem_-_Evil_Twin.mp3
/home/jules/PROJECT-THESIS/tutorial-audio_feature

### ***FRAMING THE AUDIO FILES***

In [12]:
import librosa
import numpy as np

SAMPLE_RATE:int = 16_000
# 0.025 seconds
FRAME_SIZE:int = 400       
# 0.010 seconds
STEP_SIZE:int = 160         

def frame_audio(file_audio: Path) -> np.ndarray:
    audio:np.ndarray
    audio, _ = librosa.load(file_audio, sr=SAMPLE_RATE, mono=True, dtype=np.float32)

    if audio.size < FRAME_SIZE:
        return np.empty((0, FRAME_SIZE), dtype=np.float32)

    # sliding window view creates a reference to the original audio 
    # on frame[i] it returns audio [i to i+FRAME_SIZE]
    # [::STEP_SIZE] is native python functionalit which extracts only very other entry in a list
    # ::STEP_SIZE maps 0->0, 1->STEP_SIZE*1, 2->STEP_SIZE*2 ... i -> STEP_SIZE*i
    # so frames[1] = sliding_window[STEP_SIZE] = audio[STEP_SIZE to STEP_SIZE + FRAME_SIZE]
    frames = np.lib.stride_tricks.sliding_window_view(audio, FRAME_SIZE)[::STEP_SIZE]
    return frames.astype(np.float32, copy=True)

# print(FILES_AUDIO[0])
# for x in (frame_audio(FILES_AUDIO[0])):
#     print(x) 

In [13]:
file_rows = []
for audio_dir in FILES_AUDIO:
    frames:np.ndarray = frame_audio(audio_dir)
    
    # Each row stores the source file, its one-based frame number, and its samples.
    rows:np.ndarray = np.empty((len(frames), 3), dtype=object)
    rows[:, 0] = str(audio_dir.resolve())
    rows[:, 1] = np.arange(1, len(frames) + 1)
    rows[:, 2] = list(frames)

    file_rows.append(rows)
    print(f"{audio_dir}: {len(frames)} frames")


ALL_AUDIO_FRAMES:np.ndarray = np.concatenate(file_rows, axis=0)

/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Dax-Dr-Dre-Ft-Eminem-Forgot-About-Dre-Remix-(TrendyBeatz.com).mp3: 26438 frames
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Eminem-Godzilla-ft-Juice-WRLD-(HipHopKit.com).mp3: 21267 frames
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Eminem-Mockingbird-(HipHopKit.com).mp3: 25837 frames
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Eminem-Snoop-Dogg-From-The-D-2-The-LBC-(HipHopKit.com).mp3: 28679 frames
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Eminem-ft-Joyner-Lucas-Lucky-You-(HipHopKit.com).mp3: 24504 frames
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Eminem-ft-NF-Why-(HipHopKit.com).mp3: 18908 frames
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/EminemVenom.mp3: 29584 frames
/home/jules/PROJECT-THESIS/tutorial-audio_feature_

Note: Illegal Audio-MPEG-Header 0x20526579 at offset 2664114.
Note: Trying to resync...
Note: Hit end of (available) data during resync.


/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Eminem_-_Nice_Guy_ft_Jessie_Reyez_talkglitz.tv.mp3: 14998 frames
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Eminem_-_Tone_Deaf_thinknews.com.ng.mp3: 30297 frames
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Eminem_-_Unaccommodating_ft_Young_MA_talkglitz.tv.mp3: 25969 frames
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Eminem_Alfred_(thinkNews.com.ng).mp3: 17484 frames
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Eminem_Berzerk_(thinkNews.com.ng).mp3: 24839 frames
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Eminem_Darkness_(thinkNews.com.ng).mp3: 33713 frames
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Artists/Rapper/Eminem_Discombobulated_(thinkNews.com.ng).mp3: 25261 frames
/home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/Arti

### ***SAVING THE AUDIO FRAME FILES***

In [14]:
args.output.parent.mkdir(parents=True, exist_ok=True)
np.save(args.output, ALL_AUDIO_FRAMES, allow_pickle=True)
# np.save normally works only on regular numeric arrays float32 array of shape (n_frames, frame_size).
# if the array has python lists of different lenths
# np has to store it as an object dtype array 
# which needs allow_pickle=True

# note: picking can execute files in the binary so this is a security risk

print(f"Saved {len(ALL_AUDIO_FRAMES)} frames with shape {ALL_AUDIO_FRAMES.shape} to {args.output}")
print("Columns: source_file, frame_number, samples")

Saved 1099018 frames with shape (1099018, 3) to /home/jules/PROJECT-THESIS/tutorial-audio_feature_extraction/preprocessing-01-framing.npy
Columns: source_file, frame_number, samples


### ***VERIFYING THE SAVED AUDIO FRAMES***

In [15]:
LOADED_AUDIO_FRAMES:np.ndarray = np.load(args.output, allow_pickle=True)

assert LOADED_AUDIO_FRAMES.shape == ALL_AUDIO_FRAMES.shape, (
    f"Shape changed: {ALL_AUDIO_FRAMES.shape} -> {LOADED_AUDIO_FRAMES.shape}"
)
assert LOADED_AUDIO_FRAMES.dtype == ALL_AUDIO_FRAMES.dtype, (
    f"Data type changed: {ALL_AUDIO_FRAMES.dtype} -> {LOADED_AUDIO_FRAMES.dtype}"
)
assert np.array_equal(LOADED_AUDIO_FRAMES[:, 0], ALL_AUDIO_FRAMES[:, 0]), (
    "One or more source-file paths changed after saving."
)
assert np.array_equal(LOADED_AUDIO_FRAMES[:, 1], ALL_AUDIO_FRAMES[:, 1]), (
    "One or more frame numbers changed after saving."
)
for source_file in np.unique(LOADED_AUDIO_FRAMES[:, 0]):
    frame_numbers = LOADED_AUDIO_FRAMES[
        LOADED_AUDIO_FRAMES[:, 0] == source_file, 1
    ].astype(np.int64)
    assert np.array_equal(
        frame_numbers, np.arange(1, len(frame_numbers) + 1)
    ), f"Frame sequence is not continuous for {source_file}"
assert all(
    np.array_equal(loaded_frame, original_frame)
    for loaded_frame, original_frame in zip(
        LOADED_AUDIO_FRAMES[:, 2],
        ALL_AUDIO_FRAMES[:, 2],
        strict=True,
    )
), "One or more audio frames changed after saving."

print(
    f"Integrity verified: {len(LOADED_AUDIO_FRAMES)} saved frames exactly match "
    "the frames in memory."
)

Integrity verified: 1099018 saved frames exactly match the frames in memory.
